# 文档搜索中的融合检索

## 概述

此代码实现了融合检索系统，该系统将基于向量的相似性搜索与基于关键字的 BM25 检索相结合。该方法旨在利用两种方法的优势来提高文档检索的整体质量和相关性。

## 动机

传统的检索方法通常依赖于语义理解（基于向量）或关键字匹配（BM25）。每种方法都有其优点和缺点。融合检索旨在将这些方法结合起来，创建一个更强大、更准确的检索系统，可以有效地处理更广泛的查询。

## 关键组件

1. PDF 处理和文本分块
2. 使用 FAISS 和 OpenAI 嵌入支持存储创建
3. 基于关键字检索的BM25索引创建
4. 融合BM25和向量搜索结果以实现更好的检索

## 方法详细信息

### 文档预处理

1. 使用 SentenceSplitter 加载 PDF 并将其分割成块。
2. 通过用空格替换“t”和换行符清理来清理块（可能解决特定的格式问题）。

### 支持存储创建

1. OpenAI 嵌入用于创建文本块的向量表示。
2. 根据这些嵌入创建 FAISS 存储存储，以实现高效的相似性搜索。

### BM25 指数创建

1. BM25 索引是根据用于支持存储的相同文本块创建的。
2. 这允许基于关键字的检索以及基于向量的方法。

### 查询融合检索

创建两个索引后，查询融合检索将它们组合起来以启用混合检索

## 这种方法的好处

1. 提高检索质量：通过结合语义搜索和基于关键字的搜索，系统可以捕获概念相似性和精确的关键字匹配。
2. 灵活性：`检索器_weights` 参数允许根据特定用例或查询类型调整向量搜索和关键字搜索之间的平衡。
3. 鲁棒性：组合方法可以有效地处理更广泛的查询，从而减轻单个方法的弱点。
4. 可定制性：系统可以轻松适应使用不同的支持存储或基于关键字的检索方法。

## 结论

融合检索代表了一种强大的文档搜索方法，它结合了语义理解和关键字匹配的优势。通过利用基于向量和BM25检索方法，它为信息检索任务提供了更全面、更灵活的解决方案。这种方法在概念相似性和关键词相关性都很重要的各个领域都有潜在的应用，例如学术研究、法律文档搜索或通用搜索引擎。

# 包安装和导入

下面的单元格安装运行此笔记本所需的所有必需软件包。

In [ ]:
# Install required packages
!pip install faiss-cpu llama-index python-dotenv

In [ ]:
import os
import sys
from dotenv import load_dotenv
from typing import List
from llama_index.core import Settings
from llama_index.core.readers import SimpleDirectoryReader
from llama_index.core.node_parser import SentenceSplitter
from llama_index.core.ingestion import IngestionPipeline
from llama_index.core.schema import BaseNode, TransformComponent
from llama_index.vector_stores.faiss import FaissVectorStore
from llama_index.core import VectorStoreIndex
from llama_index.llms.openai import OpenAI
from llama_index.embeddings.openai import OpenAIEmbedding
from llama_index.legacy.retrievers.bm25_retriever import BM25Retriever
from llama_index.core.retrievers import QueryFusionRetriever
import faiss

# Original path append replaced for Colab compatibility
# Load environment variables from a .env file
load_dotenv()

# Set the OpenAI API key environment variable
os.environ["OPENAI_API_KEY"] = os.getenv('OPENAI_API_KEY')

# Llamaindex global settings for llm and embeddings
EMBED_DIMENSION=512
Settings.llm = OpenAI(model="gpt-3.5-turbo", temperature=0.1)
Settings.embed_model = OpenAIEmbedding(model="text-embedding-3-small", dimensions=EMBED_DIMENSION)

### 阅读文档

In [ ]:
# Download required data files
import os
os.makedirs('data', exist_ok=True)

# Download the PDF document used in this notebook
!wget -O data/Understanding_Climate_Change.pdf https://raw.githubusercontent.com/NirDiamant/RAG_TECHNIQUES/main/data/Understanding_Climate_Change.pdf


In [ ]:
path = "data/"
reader = SimpleDirectoryReader(input_dir=path, required_exts=['.pdf'])
documents = reader.load_data()
print(documents[0])

### 创建支持存储

In [ ]:
# Create FaisVectorStore to store embeddings
fais_index = faiss.IndexFlatL2(EMBED_DIMENSION)
vector_store = FaissVectorStore(faiss_index=fais_index)

### 文本清理转换

In [ ]:
class TextCleaner(TransformComponent):
    """
    Transformation to be used within the ingestion pipeline.
    Cleans clutters from texts.
    """
    def __call__(self, nodes, **kwargs) -> List[BaseNode]:
        
        for node in nodes:
            node.text = node.text.replace('\t', ' ') # Replace tabs with spaces
            node.text = node.text.replace(' \n', ' ') # Replace paragprah seperator with spacaes
            
        return nodes

### 摄取管道

In [ ]:
# Pipeline instantiation with: 
# node parser, custom transformer, vector store and documents
pipeline = IngestionPipeline(
    transformations=[
        SentenceSplitter(),
        TextCleaner()
    ],
    vector_store=vector_store,
    documents=documents
)

# Run the pipeline to get nodes
nodes = pipeline.run()

## 搜索器

### BM25 搜索器

In [ ]:
bm25_retriever = BM25Retriever.from_defaults(
    nodes=nodes,
    similarity_top_k=2,
)

### 矢量搜索器

In [ ]:
index = VectorStoreIndex(nodes)
vector_retriever = index.as_retriever(similarity_top_k=2)

### 融合两个搜索器

In [ ]:
retriever = QueryFusionRetriever(
    retrievers=[
        vector_retriever,
        bm25_retriever
    ],
    retriever_weights=[
        0.6, # vector retriever weight
        0.4 # BM25 retriever weight
    ],
    num_queries=1, 
    mode='dist_based_score',
    use_async=False
)

关于参数

1. `num_queries`：查询融合检索器不仅可以组合检索器，还可以根据给定的查询生成多个问题。此参数控制将传递给搜索器的总查询数。因此，将其设置为 1 将禁用查询生成，并且最终的搜索器仅使用初始查询。
2. `mode`：该参数有4个选项。 
   - **reciprocal_rerank**：应用倒数排名。 （由于没有标准化，该方法不适合此类应用。因为不同的检索器会返回分数尺度）
   - **relative_score**：根据所有节点之间的最小和最大分数应用 MinMax。然后缩放到 0 到 1 之间。最后，根据 `检索器_weights` 的相关搜索器对分数进行加权。```math
      min\_score = min(scores)
      \\ max\_score = max(scores)
      ```- **dist_based_score**：与 `relative_score` 的唯一区别是 MinMax 缩放基于分数的平均值和标准差。缩放和加权是相同的。```math
       min\_score = mean\_score - 3 * std\_dev
      \\ max\_score = mean\_score + 3 * std\_dev
      ```- **简单**：此方法只是简单地取每个块的最大分数。

### 用例示例

In [ ]:
# Query
query = "What are the impacts of climate change on the environment?"

# Perform fusion retrieval
response = retriever.retrieve(query)

### 打印最终检索到的节点及其分数

In [ ]:
for node in response:
    print(f"Node Score: {node.score:.2}")
    print(f"Node Content: {node.text}")
    print("-"*100)

![](https://europe-west1-rag-techniques-views-tracker.cloudfunctions.net/rag-techniques-tracker?notebook=all-rag-techniques--fusion-retrieval-with-llamaindex)